# nano-inference — Getting Started

A hands-on first run: **serve** a diffusion LLM, **call** it, **benchmark** it, and **read the numbers**.

nano-inference is a minimal, open inference stack for **diffusion LLMs**. You bring a checkpoint; it gives
you serving (several decode modes over *one* set of weights) and reproducible benchmarks, in a few readable
files.

### Three decode modes — same weights

| `--mode` | what it is | decode block |
|---|---|---|
| `causal` | native autoregressive | — |
| `diffusion` | parallel block-diffusion (confidence-based denoising) — *not supported for Trida2.0-4B* (full-mask recipe); use `self-spec`. Kept for other block-diffusion families | `block_size: 3` = block **B=4** |
| **`self-spec`** (default) | self-speculative: diffusion draft + AR verify — lossless vs `causal`, faster per stream | block 7 / gen 4 |

### Two backends, one API

```
  your prompt                                        decode mode (causal / diffusion / self-spec)
      │                                                    │
      ▼                                                    ▼
  this notebook ──HTTP──▶  serve.py ──▶ SGLang backend  ───┘─▶ tokens ─▶ response
  (OpenAI-compatible)      (or) vllm serve + native plugin
```

- **SGLang** (`serve.py`): a thin wrapper — maps `--mode` to the backend's decode algorithm and launches
  the server. Needs the diffusion-serving SGLang backend built once (see [`sglang/README.md`](sglang/README.md))
  and a one-time **checkpoint shaping** step.
- **vLLM** (`vllm/serve_diffusion.sh`): a native plugin for stock vLLM 0.27.x — serves a raw checkpoint
  directly, no shaping (see [`vllm/README.md`](vllm/README.md)).

Both expose the same OpenAI-compatible `/v1/chat/completions` endpoint, so everything below is backend-agnostic.

## Prerequisites

1. **A GPU** and **one backend installed** — follow [`sglang/README.md`](sglang/README.md) *or*
   [`vllm/README.md`](vllm/README.md) once. This notebook does not build the backend.
2. **A compatible checkpoint** — a Hugging Face repo id or a local path. See
   [`MODEL_CONTRACT.md`](MODEL_CONTRACT.md) for what makes one compatible.
3. **Run this notebook from the `inference/` directory** (so the relative script paths resolve).
4. Python deps: `pip install -r requirements.txt` (this notebook uses `requests`; the benchmark uses
   `datasets`, and the analysis uses `matplotlib` if present).

## Configure

Set where your model is and which backend/mode to serve. If you already have a server running elsewhere,
set `LAUNCH_SERVER = False` and just point `HOST`/`PORT` at it — the call and benchmark sections work the same.

In [ ]:
import os, pathlib

# --- where's your model? (HF repo id or local path) ---
# Default: the reference checkpoint. It is private on Hugging Face during preview, so (once you have access)
# run `huggingface-cli login` (or export HF_TOKEN) before the first download.
MODEL   = os.environ.get("TRIDA_MODEL", os.environ.get("NANO_MODEL", "trillionlabs/Trida2.0-4B"))

# --- backend + mode ---
BACKEND = "sglang"      # "sglang" (serve.py) or "vllm" (native plugin)
MODE    = "self-spec"   # sglang: self-spec (default) | causal ;  vllm: self-spec (see vllm/README.md)

# --- where to reach the server ---
HOST     = "127.0.0.1"
PORT     = 30000
BASE_URL = f"http://{HOST}:{PORT}"

# Launch the server from this notebook (True), or point at an already-running one (False).
LAUNCH_SERVER = True

# For SGLang only: the interpreter that has the backend installed (see sglang/README.md, step 3).
#   export SGLANG_PYTHON=/path/to/hybrid-diffusion-eval/bin/python
SGLANG_PYTHON = os.environ.get("SGLANG_PYTHON", "python")

INF = pathlib.Path.cwd()
assert (INF / "serve.py").exists(), "Run this notebook from the inference/ directory."
print("model :", MODEL)
print("serve :", BACKEND, "/", MODE, "->", BASE_URL)

## 1. Ways to call the model

Once a server is up, it's a normal **OpenAI-compatible** endpoint. Three equivalent ways to talk to it:

| how | command | when |
|---|---|---|
| **`chat.py`** | `python chat.py "What is 17 * 24?"` | quick one-shot or interactive REPL |
| **raw HTTP** | `POST {BASE_URL}/v1/chat/completions` | from any language / this notebook |
| **`eval.py`** | `python eval.py gsm8k` | reproduce benchmark accuracy + throughput |

The rest of this notebook uses raw HTTP (via `requests`) so you can see exactly what goes over the wire.
The sampling defaults match `chat.py`/`eval.py`: `temperature 1.0 / top_p 0.95 / top_k 50`, no presence penalty,
with the thinking trace on.

## 2. Launch the server

`serve.py` (SGLang) or `serve_diffusion.sh` (vLLM) both boot an OpenAI-compatible server. The helpers below
launch it as a background process and wait until `/v1/models` responds.

**SGLang note:** a *raw* two-stream checkpoint must be **shaped** once into the architecture SGLang loads
(`tools/shape_for_sglang.py`). The next cell does that automatically for SGLang; it's a no-op for vLLM.

In [ ]:
import subprocess, sys, time, requests

proc = None          # the server process handle (kept for teardown)
served_model = MODEL # the id the server will expose (refined after launch)

def _shape_for_sglang(model_path):
    """SGLang-only: rewrite a raw checkpoint into the arch SGLang loads. Returns the shaped dir."""
    p = pathlib.Path(model_path)
    if not p.exists():
        return model_path  # HF repo id or already-shaped id — leave as-is
    import json as _json
    cfg = p / "config.json"
    if cfg.exists():
        arch = _json.loads(cfg.read_text()).get("architectures", [])
        if "Qwen3_5DLLMForConditionalGeneration" in arch:
            return model_path  # already shaped
    out = p.parent / f"{p.name}_sglang"
    subprocess.run([SGLANG_PYTHON, str(INF / "sglang" / "tools" / "shape_for_sglang.py"),
                    str(p), str(out), "--force"], check=True)
    print("shaped ->", out)
    return str(out)

def launch_server():
    env = dict(os.environ)
    log = open(f"server_{BACKEND}_{MODE}.log", "w")
    if BACKEND == "sglang":
        model = _shape_for_sglang(MODEL)
        env["SGLANG_PYTHON"] = SGLANG_PYTHON
        env["PATH"] = os.path.dirname(SGLANG_PYTHON) + os.pathsep + env.get("PATH", "")
        cmd = [sys.executable, str(INF / "serve.py"), model, "--mode", MODE, "--port", str(PORT)]
    else:  # vllm
        env["CKPT"] = MODEL
        env["PORT"] = str(PORT)
        env.setdefault("GPU_MEM_UTIL", "0.55")
        cmd = ["bash", str(INF / "vllm" / "serve_diffusion.sh")]
    print("launching:", " ".join(cmd))
    return subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)

def wait_until_ready(timeout=1200):
    """Poll /v1/models until the server answers or the process dies."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            r = requests.get(f"{BASE_URL}/v1/models", timeout=3)
            if r.ok and r.json().get("data"):
                return r.json()["data"][0]["id"]
        except requests.RequestException:
            pass
        if proc is not None and proc.poll() is not None:
            raise RuntimeError(f"server exited early (code {proc.returncode}); see server_{BACKEND}_{MODE}.log")
        time.sleep(4)
    raise TimeoutError("server did not become ready in time")

In [ ]:
if LAUNCH_SERVER:
    proc = launch_server()
    print("waiting for the server (first launch compiles kernels — can take a few minutes)...")
    served_model = wait_until_ready()
else:
    served_model = requests.get(f"{BASE_URL}/v1/models").json()["data"][0]["id"]

print("server ready — model id:", served_model)

## 3. Make a simple call

A single chat completion. `enable_thinking` toggles the model's `<think>…</think>` reasoning trace.

In [ ]:
def chat(prompt, think=True, max_tokens=512, **overrides):
    body = {
        "model": served_model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 1.0, "top_p": 0.95, "top_k": 20, "presence_penalty": 1.5,
        "chat_template_kwargs": {"enable_thinking": think},
    }
    body.update(overrides)
    r = requests.post(f"{BASE_URL}/v1/chat/completions", json=body, timeout=600)
    r.raise_for_status()
    return r.json()

resp = chat("What is 17 times 24? Answer with just the number.")
print(resp["choices"][0]["message"]["content"])

Turn the thinking trace off for a terse answer:

In [ ]:
ans = chat("Name three primary colors.", think=False)["choices"][0]["message"]["content"]
print(ans)

## 4. Run a benchmark

`eval.py` reproduces accuracy + throughput on a served endpoint (`python eval.py gsm8k`). Here we run a small
**GSM8K** subsample inline so we can keep every per-request result for analysis. It reuses `eval.py`'s exact
prompt/answer-extraction logic, so the numbers match the CLI.

Bump `N` up for a tighter estimate (the full CLI run is `python eval.py gsm8k`).

In [ ]:
import random, time
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.insert(0, str(INF))
try:
    import eval as nano_eval
    items = nano_eval.gsm8k_load()                 # needs `datasets` + network on first use
    gsm8k_prompt, gsm8k_extract = nano_eval.gsm8k_prompt, nano_eval.gsm8k_extract
    print(f"loaded GSM8K test set: {len(items)} problems")
except Exception as e:
    print("could not load GSM8K via datasets (", e, ") — using a small built-in demo set")
    # clearly-labelled fallback so the notebook still runs offline (GSM8K-style, verified answers)
    items = [
        ("A store has 24 boxes with 17 apples each. How many apples in total?", "408"),
        ("Sam reads 12 pages a day for 2 weeks. How many pages total?", "168"),
        ("A bus holds 45 people. How many people fit in 6 full buses?", "270"),
        ("Mia saves $8 per week. How much has she saved after 9 weeks?", "72"),
        ("A recipe needs 3 eggs per cake. How many eggs for 15 cakes?", "45"),
    ]
    gsm8k_prompt = lambda q: f"{q}\nPlease reason step by step, and put your final answer within \\boxed{{}}."
    import re as _re
    def gsm8k_extract(c):
        b = _re.findall(r"\\boxed\{([^}]+)\}", c)
        if b: return b[-1].replace(",", "").replace("$", "").strip()
        after = c.split("</think>")[-1] if "</think>" in c else c
        nums = _re.findall(r"[\d,]+", after)
        return nums[-1].replace(",", "") if nums else "?"

N = min(20, len(items))
random.seed(42)
sample = [items[i] for i in random.sample(range(len(items)), N)]

In [ ]:
def bench_one(item):
    q, gold = item
    t = time.time()
    try:
        r = chat(gsm8k_prompt(q), max_tokens=1024)
        c = r["choices"][0]["message"]["content"]
        dt = time.time() - t
        return dict(ok=gsm8k_extract(c) == gold, toks=r["usage"]["completion_tokens"],
                    latency=dt, err=None)
    except Exception as e:
        return dict(ok=False, toks=0, latency=time.time() - t, err=str(e))

t0 = time.time()
results = []
with ThreadPoolExecutor(max_workers=16) as pool:
    futs = [pool.submit(bench_one, it) for it in sample]
    for i, f in enumerate(as_completed(futs), 1):
        results.append(f.result())
        if i % max(N // 5, 1) == 0:
            acc = sum(r["ok"] for r in results) / len(results) * 100
            print(f"  {i}/{N}  running acc={acc:.1f}%  ({time.time()-t0:.0f}s)")
wall = time.time() - t0
print(f"done: {N} problems in {wall:.0f}s")

## 5. Analyze the result

Accuracy, throughput, and latency — the three numbers you care about — plus a look at the per-request spread.

In [ ]:
n        = len(results)
correct  = sum(r["ok"] for r in results)
tokens   = sum(r["toks"] for r in results)
errors   = sum(r["err"] is not None for r in results)
latencies = sorted(r["latency"] for r in results if r["err"] is None)

def pct(p):
    if not latencies: return float('nan')
    return latencies[min(len(latencies)-1, int(p/100*len(latencies)))]

print("── benchmark summary ─────────────────────────")
print(f"accuracy        : {correct}/{n} = {correct/n*100:.1f}%")
print(f"throughput      : {tokens/wall:.0f} tok/s (aggregate, {tokens} tokens / {wall:.0f}s)")
print(f"latency  p50    : {pct(50):.1f}s")
print(f"latency  p90    : {pct(90):.1f}s")
print(f"errors          : {errors}")

In [ ]:
# Optional visual: latency distribution + accuracy. Skips cleanly if matplotlib is absent.
try:
    import matplotlib.pyplot as plt
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4))
    a.hist(latencies, bins=min(12, max(3, n // 2)), color="#4C78A8", edgecolor="white")
    a.set_title("per-request latency"); a.set_xlabel("seconds"); a.set_ylabel("requests")
    b.bar(["correct", "wrong"], [correct, n - correct], color=["#54A24B", "#E45756"])
    b.set_title(f"accuracy — {correct/n*100:.0f}%"); b.set_ylabel("requests")
    fig.tight_layout(); plt.show()
except Exception as e:
    print("(matplotlib not available — skipping plot:", e, ")")

Reading the numbers:

- **accuracy** — should track the model's published GSM8K on this decode mode; a big drop means the mode/config
  or checkpoint isn't matched (see [`MODEL_CONTRACT.md`](MODEL_CONTRACT.md)).
- **throughput** — aggregate tok/s across the concurrent workers; raise `max_workers` and `N` to load the server.
- **latency spread** — diffusion decode does several denoising passes per block, so per-request latency depends on
  how many passes the `threshold` gate triggers. Compare `--mode causal` vs `diffusion` vs `self-spec` on the same
  prompts to see the trade-off.

## Clean up

Shut the server down when you're done (only if this notebook launched it).

In [ ]:
if proc is not None and proc.poll() is None:
    proc.terminate()
    try:
        proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("server stopped")
else:
    print("no server owned by this notebook (or already stopped)")

## Next steps

- **Full benchmark:** `python eval.py gsm8k` (whole test set) or `python eval.py mmlu_pro --num-problems 1000`;
  instruction-following is `python eval_ifeval.py`.
- **Compare modes:** re-run this notebook with `MODE = "causal"`, `"diffusion"`, `"self-spec"` and compare
  accuracy vs latency on the same prompts.
- **Other model families:** `sdar`, `llada2-0`, `llada2-1-speed/quality` — see the README's model table.
- **Your own checkpoint:** point `MODEL` at it; read [`MODEL_CONTRACT.md`](MODEL_CONTRACT.md) first.